# CS50P - Semana 9: Et Cetera

---

En esta unidad final exploramos una colección de características y técnicas de Python que, aunque no son estrictamente necesarias para resolver problemas básicos, son fundamentales para escribir código profesional, eficiente y legible.

### Objetivos de Aprendizaje
* Manejar colecciones de datos únicos con `set`.
* Comprender el alcance de las variables con `global`.
* Implementar validaciones estáticas con **Type Hints** y `mypy`.
* Crear interfaces de línea de comandos robustas con `argparse`.
* Dominar el desempaquetado de datos (`*args` y `**kwargs`).
* Escribir código funcional con `map`, `filter` y **List Comprehensions**.
* Optimizar el uso de memoria con **Generadores** y `yield`.

---

### Índice
1. [Sets (Conjuntos)](#sets)
2. [Global Variables (Variables Globales)](#global)
3. [Constants (Constantes)](#constants)
4. [Type Hints & mypy](#type-hints)
5. [Docstrings](#docstrings)
6. [argparse](#argparse)
7. [Unpacking (Desempaquetado)](#unpacking)
8. [*args y **kwargs](#args-kwargs)
9. [map](#map)
10. [List Comprehensions](#list-comprehensions)
11. [filter](#filter)
12. [Dictionary Comprehensions](#dictionary-comprehensions)
13. [enumerate](#enumerate)
14. [Generadores e Iteradores](#generators)

---

<h2 id="sets">1. Sets (Conjuntos)</h2>

Un `set` es una colección de datos que tiene dos características principales:
1. **No permite duplicados**: Cada elemento es único.
2. **No tiene orden**: Los elementos no se guardan en una posición específica.

Es ideal para cuando solo nos interesa saber qué elementos hay, sin importar cuántas veces aparecen o en qué orden.

#### Ejemplo: Filtrar estudiantes en casas de Hogwarts
Imagina que tenemos una lista de diccionarios de estudiantes y queremos obtener una lista única de las casas (Houses) presentes.

In [ ]:
students = [
    {"name": "Hermione", "house": "Gryffindor"},
    {"name": "Harry", "house": "Gryffindor"},
    {"name": "Ron", "house": "Gryffindor"},
    {"name": "Draco", "house": "Slytherin"},
    {"name": "Padma", "house": "Ravenclaw"},
]

# Método tradicional con listas
houses_list = []
for student in students:
    if student["house"] not in houses_list:
        houses_list.append(student["house"])

# Método eficiente con Sets
houses_set = set()
for student in students:
    houses_set.add(student["house"])

print("Lista única (tradicional):", sorted(houses_list))
print("Conjunto único (set):", sorted(houses_set))

En el código anterior:
* `set()` crea un conjunto vacío.
* `.add()` intenta añadir un elemento. Si el elemento ya existe (como "Gryffindor"), el set simplemente lo ignora.
* Al final, obtenemos una colección de valores únicos de forma automática.

> **Nota:** Al usar `sorted()` sobre un set, Python nos devuelve una **lista** ordenada, lo cual es muy útil para presentar los resultados.

---


<h2 id="global">2. Variables Globales (Global Variables)</h2>

En muchos lenguajes de programación existe la noción de **variables globales**: variables que se definen fuera de cualquier función y que son accesibles desde cualquier rincón del código.

En Python, podemos aprovechar esta capacidad de forma sencilla para **leer** valores.

#### Ejemplo 1: Acceso de lectura
Si solo necesitamos leer el valor, Python nos permite acceder a la variable global sin problemas:

In [ ]:
balance = 0

def main():
    # Podemos leer 'balance' porque vive en el scope global
    print("Balance:", balance)

if __name__ == "__main__":
    main()

### El Problema: Modificar no es tan fácil

Aunque leer es sencillo, intentar **modificar** una variable global dentro de una función sin avisar a Python provoca un error. El intérprete asume que si estás asignando un valor a una variable dentro de una función, esa variable debe ser **local**.

Observemos qué sucede si intentamos actualizar el balance sin herramientas adicionales:

In [ ]:
balance = 0

def main():
    print("Balance inicial:", balance)
    deposit(100)
    withdraw(50)
    print("Balance final:", balance)

def deposit(n):
    # Intentamos reasignar balance
    balance += n

def withdraw(n):
    # Intentamos reasignar balance
    balance -= n

if __name__ == "__main__":
    # ¡CUIDADO! Esto lanzará un UnboundLocalError
    try:
        main()
    except UnboundLocalError as e:
        print(f"\nError capturado: {e}")

#### ¿Por qué falló?
Al escribir `balance += n`, Python ve una asignación. Por seguridad, asume que `balance` es una variable local que aún no ha sido creada dentro de la función, lanzando un **`UnboundLocalError`**.

#### La Solución: La palabra clave `global`
Para interactuar y modificar una variable global, debemos usar explícitamente la palabra clave `global`. Esto le indica a la función: *"No crees una variable local, usa la que está definida arriba de todo"*.

In [ ]:
balance = 0

def main():
    print("Balance inicial:", balance)
    deposit(100)
    withdraw(50)
    print("Balance final:", balance)

def deposit(n):
    global balance
    balance += n

def withdraw(n):
    global balance
    balance -= n

if __name__ == "__main__":
    main()

### Una solución más limpia: Clases (POO)

Utilizando lo aprendido en la **Semana 8**, podemos ver que las variables globales suelen ser un síntoma de que necesitamos un objeto. Las clases nos permiten resolver este problema de forma más organizada, ya que las variables de instancia son accesibles para todos los métodos de la clase a través de `self`.

Esto evita "ensuciar" el espacio global del programa y encapsula la lógica.

In [ ]:
class Account:
    def __init__(self):
        self._balance = 0

    @property
    def balance(self):
        return self._balance

    def deposit(self, n):
        self._balance += n

    def withdraw(self, n):
        self._balance -= n

def main():
    account = Account()
    print("Balance inicial:", account.balance)
    account.deposit(100)
    account.withdraw(50)
    print("Balance final:", account.balance)

if __name__ == "__main__":
    main()

> **Regla general:** Las variables globales deben usarse con mucha moderación, o preferiblemente, ¡no usarse en absoluto! El uso de clases o el paso de argumentos entre funciones suele dar como resultado un código mucho más fácil de mantener y menos propenso a errores inesperados.

---

<h2 id="constants">3. Constantes (Constants)</h2>

En otros lenguajes de programación, existe la noción de **constantes**: variables cuyo valor no puede ser modificado una vez asignado. Las constantes permiten programar de forma defensiva, reduciendo las posibilidades de que valores importantes sean alterados accidentalmente.

#### Convención de Nomenclatura
En Python, denotamos las constantes utilizando nombres de variables en **MAYÚSCULAS**, colocándolas usualmente en la parte superior de nuestro código.

In [ ]:
# Definición de la constante
MEOWS = 3

for _ in range(MEOWS):
    print("meow")

### El Sistema de Honor (The Honor System)

Aunque el ejemplo anterior **parece** una constante, en realidad, ¡Python no tiene ningún mecanismo interno para evitar que cambiemos ese valor! 

En Python, funcionamos bajo el **sistema de honor**: si ves un nombre de variable escrito totalmente en mayúsculas, simplemente **no lo cambies**. Es una señal para otros programadores de que ese valor debe tratarse como inmutable.

### "Constantes" de Clase

También podemos crear constantes dentro de una clase. Esto es útil para agrupar valores fijos que están relacionados directamente con un objeto o concepto específico.

Debido a que se definen fuera de cualquier método de clase particular, todos los métodos tienen acceso a ese valor a través del nombre de la clase (ej: `NombreClase.CONSTANTE`).

In [ ]:
class Cat:
    # Constante de clase
    MEOWS = 3

    def meow(self):
        # Accedemos a la constante a través del nombre de la clase
        for _ in range(Cat.MEOWS):
            print("meow")

cat = Cat()
cat.meow()

En este caso, `MEOWS` actúa como un valor compartido. Al estar definido fuera de los métodos, no pertenece a una instancia específica (como `self.name`), sino a la clase `Cat` en general. Esto permite centralizar configuraciones importantes en un solo lugar.

---

<h2 id="type-hints">4. Type Hints</h2>

En muchos lenguajes de programación, es obligatorio expresar explícitamente qué tipo de variable se quiere usar. Como hemos visto a lo largo del curso, Python no requiere esta declaración explícita, pero es una **buena práctica** para asegurar que todas las variables sean del tipo correcto.

Para ayudarnos en esta tarea, utilizamos **mypy**, un programa que verifica de forma estática que tus variables respeten los tipos declarados.

#### Instalación
Puedes instalar `mypy` ejecutando el siguiente comando en tu terminal:
`pip install mypy`

#### El problema del tipado dinámico
Observa el siguiente ejemplo donde esperamos un entero, pero recibimos un string:

In [ ]:
def meow(n):
    for _ in range(n):
        print("meow")

# input() siempre devuelve un string (str)
number = input("Number: ")
# Esto causará un TypeError al intentar usarse en range()
try:
    meow(number)
except TypeError as e:
    print(f"\n❌ Error: {e}")

### Inyectando pistas (Hints)

Podemos añadir un **Type Hint** para indicarle a Python qué tipo de variable espera la función `meow`. 

Aunque el programa seguirá lanzando un error al ejecutarse si el tipo es incorrecto, usar la herramienta `mypy` en la terminal (`mypy meows.py`) nos proporcionará una guía clara sobre cómo solucionar el error antes de correr el código.

In [ ]:
def meow(n: int): # Indicamos que n debe ser int
    for _ in range(n):
        print("meow")

number = input("Number: ")
meow(number) 
# Si ejecutaras 'mypy' sobre este archivo, te avisaría del conflicto

### Anotación completa

No solo las funciones pueden tener pistas; también podemos anotar las variables. Al ejecutar `mypy`, recibiremos un feedback mucho más específico. La solución final para que `mypy` no reporte errores es realizar un **cast** (conversión) de la entrada a un entero.

In [ ]:
def meow(n: int):
    for _ in range(n):
        print("meow")

# Convertimos explícitamente a int para satisfacer el Type Hint
number: int = int(input("Number: "))
meow(number)

### Anotando el valor de retorno

Podemos ir un paso más allá y anotar qué devuelve una función utilizando la sintaxis `-> Type`.

* Si una función solo tiene un efecto secundario (como imprimir) y no devuelve nada, usamos `-> None`.
* Si intentamos almacenar el resultado de una función `-> None` en una variable tipada como `str`, `mypy` nos alertará del error.

In [ ]:
def meow(n: int) -> None: # Indicamos que no hay valor de retorno
    for _ in range(n):
        print("meow")

number: int = 3
# Intentar guardar el retorno (None) en una variable str
# mypy detectará que 'meow' devuelve None, no str
meows: str = meow(number) 
print(meows)

In [ ]:
def meow(n: int) -> str:
    # Retornamos el string repetido
    return "meow\n" * n

number: int = 3
meows: str = meow(number) # Ahora los tipos coinciden perfectamente
print(meows, end="")


Para profundizar más, puedes consultar:
* [Documentación oficial de Python sobre Type Hints](https://docs.python.org/3/library/typing.html)
* [Documentación oficial de mypy](https://mypy.readthedocs.io/)

---

<h2 id="docstrings">5. Docstrings</h2>

Una forma estándar de comentar el propósito de una función es utilizar un **docstring**. A diferencia de los comentarios normales (`#`), los docstrings se encierran entre triples comillas dobles (`"""`) y están diseñados para describir qué hace la función, sus parámetros y lo que devuelve.



#### Ejemplo básico:

In [ ]:
def meow(n):
    """Meow n times."""
    return "meow\n" * n

number = int(input("Number: "))
meows = meow(number)
print(meows, end="")

### Estandarización de la documentación

Puedes utilizar los docstrings para estandarizar la forma en que documentas las características de una función. Existen diferentes formatos (como Google Style, NumPy o ReST), pero el objetivo es el mismo: describir los parámetros, los tipos de datos, las excepciones que puede lanzar y el valor de retorno.

#### Ventajas de usar Docstrings:
* **Legibilidad**: Cualquiera puede entender qué hace la función sin leer el código interno.
* **Autodocumentación**: Herramientas como **Sphinx** pueden leer estos docstrings y generar automáticamente páginas web o archivos PDF con la documentación completa de tu proyecto.
* **Ayuda en el IDE**: Al pasar el ratón sobre la función en VS Code, verás esta documentación en una ventana emergente.

In [ ]:
def meow(n):
    """
    Meow n times.

    :param n: Number of times to meow
    :type n: int
    :raise TypeError: If n is not an int
    :return: A string of n meows, one per line
    :rtype: str
    """
    if not isinstance(n, int):
        raise TypeError("n must be an integer")
    return "meow\n" * n

number = int(input("Number: "))
meows = meow(number)
print(meows, end="")


Para aprender más sobre las convenciones de documentación, puedes consultar:
* [PEP 257 - Docstring Conventions](https://peps.python.org/pep-0257/)
* [Guía de herramientas de documentación como Sphinx](https://www.sphinx-doc.org/)

---

<h2 id="argparse">6. argparse</h2>

Cuando queremos que nuestro programa acepte argumentos directamente desde la terminal (línea de comandos), solemos empezar usando `sys.argv`. Sin embargo, a medida que el programa se vuelve más complejo, manejar manualmente la validación de estos argumentos se convierte en una pesadilla de condicionales.

#### La aproximación manual con `sys.argv`
Imagina que queremos que nuestro programa maúlle un número específico de veces usando la bandera `-n`.

In [ ]:
import sys

# Comprobamos la longitud y el contenido de los argumentos manualmente
if len(sys.argv) == 1:
    print("meow")
elif len(sys.argv) == 3 and sys.argv[1] == "-n":
    try:
        n = int(sys.argv[2])
        for _ in range(n):
            print("meow")
    except ValueError:
        print("Error: NUMBER must be an integer")
else:
    print("usage: meows.py [-n NUMBER]")

### La solución profesional: `argparse`

`argparse` es una librería estándar que maneja de forma automática el análisis de cadenas complicadas en la línea de comandos. 

Funciona mediante la creación de un objeto **`parser`** (de la clase `ArgumentParser`), al cual le indicamos qué argumentos esperamos. Al final, el método `parse_args()` se encarga de verificar que el usuario haya ingresado todo correctamente.

In [ ]:
import argparse

# 1. Creamos el objeto parser
parser = argparse.ArgumentParser()

# 2. Definimos qué argumento esperamos (-n)
parser.add_argument("-n")

# 3. Analizamos los argumentos recibidos
args = parser.parse_args(args=["-n", "3"])

# Accedemos al valor a través de args.n
# Nota: Por defecto, argparse guarda los valores como strings
if args.n:
    for _ in range(int(args.n)):
        print("meow")
else:
    print("meow")

### Documentación automática

Una de las mayores ventajas de `argparse` es que genera automáticamente mensajes de ayuda. Al añadir los parámetros `description` y `help`, permitimos que el usuario ejecute el programa con las banderas `-h` o `--help` para ver las instrucciones de uso sin que tengamos que escribir una sola línea de `print` extra.

In [ ]:
import argparse

# Añadimos una descripción general del programa
parser = argparse.ArgumentParser(description="Meow like a cat")

# Añadimos ayuda específica para el argumento
parser.add_argument("-n", help="number of times to meow")

args = parser.parse_args(args=["-n", "5"])

if args.n:
    for _ in range(int(args.n)):
        print("meow")

### Programación Defensiva: `type` y `default`

Podemos refinar aún más nuestro parser indicándole:
1. **`type`**: El tipo de dato esperado (ej: `int`). `argparse` hará la conversión por nosotros y lanzará un error si el usuario introduce texto.
2. **`default`**: Un valor por defecto en caso de que el usuario no proporcione el argumento.

In [ ]:
import argparse

parser = argparse.ArgumentParser(description="Meow like a cat")

# Configuramos el tipo como int y un valor por defecto de 1
parser.add_argument("-n", default=1, help="number of times to meow", type=int)

args = parser.parse_args(args=["-n", "3"])

# Ya no necesitamos convertir a int() ni comprobar si existe, gracias a default
for _ in range(args.n):
    print("meow")


`argparse` es extremadamente flexible y permite manejar argumentos posicionales, banderas booleanas y mucho más.
* [Documentación oficial de Python sobre argparse](https://docs.python.org/3/library/argparse.html)

---

<h2 id="unpacking">7. Unpacking (Desempaquetado)</h2>

¿No sería genial poder dividir una sola variable en varias de forma automática? Python nos permite hacer esto de manera muy intuitiva. 

#### Desempaquetado básico
Un ejemplo común es cuando pedimos un nombre completo y queremos separar el nombre del apellido usando `.split()`.

In [9]:
# Dividimos la entrada por el espacio y asignamos a dos variables
# Usamos '_' como nombre de variable para indicar que el segundo valor no nos interesa
first, _ = input("What's your name? ").split(" ")
print(f"hello, {first}")

ValueError: not enough values to unpack (expected 2, got 1)

### Desempaquetado de secuencias con `*`

Imagina que tenemos una función que calcula el valor total de monedas en el mundo de Harry Potter (*Galleons, Sickles y Knuts*). Si tenemos nuestras monedas guardadas en una lista, pasar los valores uno por uno es verboso y propenso a errores.

Aquí es donde entra el operador `*`. Este operador **desempaqueta** una secuencia (como una lista o tupla) y pasa cada uno de sus elementos como un argumento individual a la función.

In [ ]:
def total(galleons, sickles, knuts):
    return (galleons * 17 + sickles) * 29 + knuts

coins = [100, 50, 25]

# Forma manual y verbosa:
# print(total(coins[0], coins[1], coins[2]), "Knuts")

# Forma elegante con Unpacking:
print(total(*coins), "Knuts")

### Desempaquetado de Diccionarios con `**`

Cuando trabajamos con nombres y valores (claves y datos), los diccionarios son nuestros mejores amigos. Sin embargo, una función normal espera argumentos individuales, no un objeto diccionario completo.

El operador `**` permite desempaquetar un diccionario, pasando las **claves como nombres de los argumentos** y los **valores como sus datos correspondientes**.

In [ ]:
def total(galleons, sickles, knuts):
    return (galleons * 17 + sickles) * 29 + knuts

coins = {"galleons": 100, "sickles": 50, "knuts": 25}

# El operador ** mapea automáticamente "galleons" a galleons, etc.
print(total(**coins), "Knuts")

El desempaquetado con `**` es extremadamente potente porque:
1. Permite pasar los argumentos en cualquier orden (siempre que las llaves coincidan con los nombres de los parámetros).
2. Hace que el código sea mucho más dinámico y fácil de leer cuando manejas muchos parámetros de configuración.

---

<h2 id="args-kwargs">8. *args y **kwargs</h2>

¿Alguna vez te has preguntado cómo la función `print` puede aceptar tantos argumentos como quieras? Si miramos su documentación oficial, vemos algo como esto:

`print(*objects, sep=' ', end='\n', ...)`

* **`*args`**: Se refiere a los argumentos posicionales (como `"Hello", "World"`). El asterisco le dice a Python que empaquete todos esos valores en una **tupla**.
* **`**kwargs`**: Se refiere a los "Keyword Arguments" o argumentos de nombre (como `end=""` o `sep="-"`). El doble asterisco los empaqueta en un **diccionario**.

### Argumentos Posicionales con `*args`

Podemos definir nuestra propia función para que acepte un número desconocido de valores posicionales. Al usar `*args`, tratamos todos los valores que reciba la función como una secuencia.

In [ ]:
def f(*args, **kwargs):
    # args se comporta como una tupla
    print("Positional arguments:", args)

# Pasamos tres números individuales
f(100, 50, 25)

# Pasamos strings
f("Harry", "Ron", "Hermione")

### Argumentos de Palabra Clave con `**kwargs`

De la misma manera, podemos capturar argumentos que vienen con un nombre asignado. Estos se guardan internamente como un diccionario donde la clave es el nombre del argumento y el valor es el dato proporcionado.

In [ ]:
def f(*args, **kwargs):
    # kwargs se comporta como un diccionario
    print("Named arguments (Keyword):", kwargs)

# Pasamos valores con nombre
f(galleons=100, sickles=50, knuts=25)

# Podemos combinar ambos en la misma función

En resumen:
* `*args` permite que `print(*objects)` tome cualquier número de objetos para imprimir.
* `**kwargs` permite configurar opciones adicionales como `sep` o `end`.

Esta técnica es fundamental cuando escribes "wrappers" (funciones que envuelven a otras funciones) o cuando trabajas con librerías donde no sabes cuántos parámetros querrá configurar el usuario.


Para más detalles, puedes consultar la documentación oficial de [Python: print](https://docs.python.org/3/library/functions.html#print).

---

<h2 id="map">9. map</h2>

A lo largo del curso hemos pasado por la programación **procedural** (paso a paso) y la **orientada a objetos**. Ahora exploramos pistas de la **programación funcional**, donde las funciones pueden pasarse como argumentos a otras funciones.

#### El punto de partida: La función `yell`
Imagina una función simple que "grita" (pone en mayúsculas) una palabra. Esto tiene un "efecto secundario": imprime en pantalla pero no devuelve nada.

In [ ]:
def main():
    yell("This is CS50")

def yell(word):
    print(word.upper())

if __name__ == "__main__":
    main()

### Iterando sobre una lista

¿Qué pasa si queremos gritar una lista ilimitada de palabras? Una aproximación inicial sería pasar una lista y usar un bucle `for` para acumular los resultados.

In [ ]:
def main():
    yell(["This", "is", "CS50"])

def yell(words):
    uppercased = []
    for word in words:
        uppercased.append(word.upper())
    # Usamos * para desempaquetar la lista al imprimir
    print(*uppercased)

if __name__ == "__main__":
    main()

Podemos hacer que la llamada a la función sea más limpia eliminando los corchetes de la lista, utilizando `*words` para aceptar cualquier número de argumentos posicionales.

In [ ]:
def main():
    yell("This", "is", "CS50")

def yell(*words):
    uppercased = []
    for word in words:
        uppercased.append(word.upper())
    print(*uppercased)

if __name__ == "__main__":
    main()

### Mapeando funciones

La función `map` nos permite aplicar una función a cada elemento de una secuencia de valores de forma directa y elegante.



`map` toma dos argumentos:
1. **La función** que queremos aplicar (sin ejecutarla, solo el nombre).
2. **La secuencia** (lista, tupla, etc.) a la que queremos aplicar dicha función.

De esta forma, todos los elementos en `words` se entregarán a `str.upper` y los resultados se guardarán en `uppercased`.

In [ ]:
def main():
    yell("This", "is", "CS50")

def yell(*words):
    # Aplicamos str.upper a cada elemento en words
    uppercased = map(str.upper, words)
    print(*uppercased)

if __name__ == "__main__":
    main()


`map` devuelve un iterador, por lo que es muy eficiente en términos de memoria.
* [Documentación oficial de Python sobre map](https://docs.python.org/3/library/functions.html#map)

---

<h2 id="list-comprehensions">10. List Comprehensions</h2>

Las **List Comprehensions** te permiten crear una lista "al vuelo" en una sola línea elegante. Es una alternativa más "pitónica" al uso de `map` cuando la lógica de transformación es sencilla.



#### Ejemplo: Gritando con List Comprehension
En lugar de usar `map`, podemos escribir una expresión de Python dentro de corchetes. Para cada argumento en la secuencia, aplicamos `.upper()`.

In [ ]:
def main():
    yell("This", "is", "CS50")

def yell(*words):
    # Estructura: [expresión for elemento in iterable]
    uppercased = [arg.upper() for arg in words]
    print(*uppercased)

if __name__ == "__main__":
    main()

### Filtrado Dinámico

El verdadero poder de las List Comprehensions aparece cuando introducimos **condicionales**. Esto nos permite crear una lista que solo contenga los elementos que cumplen ciertos criterios.

#### El problema: Buscar Gryffindors
Imagina que tenemos una lista de diccionarios y queremos extraer solo los nombres de quienes pertenecen a la casa "Gryffindor".

In [ ]:
students = [
    {"name": "Hermione", "house": "Gryffindor"},
    {"name": "Harry", "house": "Gryffindor"},
    {"name": "Ron", "house": "Gryffindor"},
    {"name": "Draco", "house": "Slytherin"},
]

gryffindors = []
for student in students:
    if student["house"] == "Gryffindor":
        gryffindors.append(student["name"])

for gryffindor in sorted(gryffindors):
    print(gryffindor)

### La solución en una sola línea

Podemos simplificar todo el proceso de filtrado y extracción de datos utilizando una List Comprehension que incluya un `if` al final. 

La sintaxis es: 
`[valor_a_guardar for elemento in coleccion if condicion]`

Esto no solo es más corto, sino que a menudo es más rápido de ejecutar.

In [ ]:
students = [
    {"name": "Hermione", "house": "Gryffindor"},
    {"name": "Harry", "house": "Gryffindor"},
    {"name": "Ron", "house": "Gryffindor"},
    {"name": "Draco", "house": "Slytherin"},
]

# Creamos la lista filtrada directamente
gryffindors = [
    student["name"] for student in students if student["house"] == "Gryffindor"
]

for gryffindor in sorted(gryffindors):
    print(gryffindor)

---

<h2 id="filter">11. filter</h2>

La función `filter` de Python nos permite obtener un subconjunto de una secuencia para el cual una condición específica es verdadera (`True`). 

Al igual que `map`, `filter` toma dos argumentos:
1. **Una función**: Que debe devolver un valor booleano (`True` o `False`).
2. **Una secuencia**: Los datos que queremos filtrar.



#### Ejemplo: Filtrando con una función definida
Definiremos una función `is_gryffindor` que actúe como nuestro criterio de filtrado.

In [ ]:
students = [
    {"name": "Hermione", "house": "Gryffindor"},
    {"name": "Harry", "house": "Gryffindor"},
    {"name": "Ron", "house": "Gryffindor"},
    {"name": "Draco", "house": "Slytherin"},
]

def is_gryffindor(s):
    # Retorna True si es de Gryffindor, False si no
    return s["house"] == "Gryffindor"

# filter aplica 'is_gryffindor' a cada elemento de 'students'
gryffindors = filter(is_gryffindor, students)

# Imprimimos ordenando por nombre
for gryffindor in sorted(gryffindors, key=lambda s: s["name"]):
    print(gryffindor["name"])

### Filtros con Funciones Lambda

A veces, crear una función con `def` solo para usarla una vez dentro de un `filter` es demasiado trabajo. Podemos usar una **función lambda** (una función anónima de una sola línea) para que el código sea mucho más compacto.

In [ ]:
students = [
    {"name": "Hermione", "house": "Gryffindor"},
    {"name": "Harry", "house": "Gryffindor"},
    {"name": "Ron", "house": "Gryffindor"},
    {"name": "Draco", "house": "Slytherin"},
]

# Usamos lambda en lugar de definir una función externa
gryffindors = filter(lambda s: s["house"] == "Gryffindor", students)

for gryffindor in sorted(gryffindors, key=lambda s: s["name"]):
    print(gryffindor["name"])

* **List Comprehension**: Es más legible para la mayoría de los programadores de Python y suele ser un poco más rápida.
* **filter**: Es excelente cuando ya tienes una función compleja definida en otra parte de tu código que quieres reutilizar para filtrar.


* [Documentación oficial de Python sobre filter](https://docs.python.org/3/library/functions.html#filter)

---

<h2 id="dictionary-comprehensions">12. Dictionary Comprehensions</h2>

Podemos aplicar la misma lógica de las *list comprehensions* a los diccionarios. Esta técnica es extremadamente útil cuando queremos transformar una lista de elementos en una estructura de datos más rica o mapear etiquetas a valores específicos.

#### El enfoque tradicional (Bucle for)
Imagina que tenemos una lista de nombres y queremos generar una lista de diccionarios, asignando a cada estudiante la casa "Gryffindor".

In [ ]:
students = ["Hermione", "Harry", "Ron"]
gryffindors = []

for student in students:
    gryffindors.append({"name": student, "house": "Gryffindor"})

print(gryffindors)

### Diccionarios dentro de una List Comprehension

Podemos simplificar el código anterior metiendo la estructura del diccionario directamente dentro de una *list comprehension*. Aquí, Python crea un nuevo diccionario para cada `student` en la lista original.

In [ ]:
students = ["Hermione", "Harry", "Ron"]

# Creamos la lista de diccionarios en una sola línea
gryffindors = [{"name": student, "house": "Gryffindor"} for student in students]

print(gryffindors)

### La Comprensión de Diccionario real

Si lo que buscamos no es una lista de diccionarios, sino **un único diccionario** donde los nombres sean las *claves* y las casas los *valores*, usamos la sintaxis de llaves `{}` con la relación `clave: valor`.

Esta es la forma más eficiente de mapear datos en Python.

In [ ]:
students = ["Hermione", "Harry", "Ron"]

# Sintaxis: {clave: valor for elemento in iterable}
gryffindors = {student: "Gryffindor" for student in students}

print(gryffindors)

<h2 id="enumerate">13. enumerate</h2>

A veces no solo queremos acceder a los elementos de una lista, sino que también necesitamos conocer su **índice** o posición (por ejemplo, para crear un ranking o un listado numerado).

#### El enfoque tradicional (C-style)
La forma clásica de hacer esto es usar `range(len())` para generar índices numéricos y luego acceder a la lista mediante corchetes. Aunque funciona, es menos legible y más propenso a errores.

In [ ]:
students = ["Hermione", "Harry", "Ron"]

for i in range(len(students)):
    # Sumamos 1 para que el ranking no empiece en 0
    print(i + 1, students[i])

### Iterando con índices de forma Pitónica

La función `enumerate` nos permite obtener tanto el índice como el valor de cada elemento de forma simultánea. Al iterar, `enumerate` devuelve una **tupla** que podemos desempaquetar directamente en dos variables (usualmente `i` para el índice y el nombre del elemento).



Es mucho más limpio porque eliminamos la necesidad de acceder a la lista por índice manualmente (`students[i]`).

In [ ]:
students = ["Hermione", "Harry", "Ron"]

# i recibe el índice (0, 1, 2...) y student recibe el valor
for i, student in enumerate(students):
    print(i + 1, student)


Un truco extra: `enumerate` acepta un segundo argumento opcional llamado `start` si quieres empezar a contar desde un número diferente a cero sin sumar manualmente.
* [Documentación oficial de Python sobre enumerate](https://docs.python.org/3/library/functions.html#enumerate)

---

<h2 id="generators">14. Generadores e Iteradores (Generators and Iterators)</h2>

En Python, existe una forma de proteger tu sistema contra el agotamiento de recursos cuando los problemas que intentas resolver se vuelven demasiado grandes. 

Para ilustrar esto, usemos la analogía de "contar ovejas" para dormir. Empecemos con un programa básico que imprime ovejas (representadas por el emoji 🐑).

In [ ]:
def main():
    n = int(input("¿Cuántas ovejas quieres contar? "))
    for i in range(n):
        print(sheep(i))

def sheep(n):
    return "🐑" * n

if __name__ == "__main__":
    main()

### ¿Qué pasa con los datos grandes?

En el ejemplo anterior, la función `main` hacía el trabajo de iterar. Pero, ¿qué pasa si queremos que la función `sheep` genere todo el "rebaño" primero y nos lo devuelva?

Si intentamos generar un rebaño de **1,000,000** de ovejas y guardarlas en una lista antes de devolverlas, es muy probable que el programa se congele o se bloquee. Esto sucede porque el ordenador intenta reservar una cantidad masiva de memoria RAM para guardar todos los elementos a la vez.

In [ ]:
def main():
    n = int(input("¿Cuántas ovejas? "))
    # Esta función intentará crear la lista completa en memoria antes de empezar el bucle
    for s in sheep(n):
        print(s)

def sheep(n):
    flock = []
    for i in range(n):
        flock.append("🐑" * i)
    return flock

if __name__ == "__main__":
    # Prueba con 10 o 100, pero ten cuidado con números gigantes aquí
    main()

### Entra el Generador (`yield`)

Un **generador** resuelve este problema devolviendo los resultados poco a poco, uno a la vez. En lugar de usar `return` (que termina la función y devuelve todo), usamos **`yield`**.

`yield` permite que la función entregue un valor, se "pause" para dejar que el bucle `for` haga su trabajo, y luego continúe justo donde se quedó para generar el siguiente valor.

In [ ]:
def main():
    n = int(input("¿Cuántas ovejas con generador? "))
    # El bucle for pide una oveja, el generador se la da y se pausa.
    # Así, nunca tenemos más de una oveja a la vez en memoria.
    for s in sheep(n):
        print(s)

def sheep(n):
    for i in range(n):
        # Entregamos la oveja actual y pausamos la función
        yield "🐑" * i

if __name__ == "__main__":
    main()

### Conclusión de Et Cetera

Los generadores son herramientas de **evaluación perezosa** (lazy evaluation). No calculan el siguiente valor hasta que no es estrictamente necesario. Esto permite trabajar con conjuntos de datos teóricamente infinitos con un uso de memoria mínimo.

---
* [Documentación oficial sobre Generadores](https://docs.python.org/3/howto/functional.html#generators)
* [Documentación oficial sobre Iteradores](https://docs.python.org/3/tutorial/classes.html#iterators)